In [18]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from llama_index.llms.cohere import Cohere
from llama_index.core.llms import ChatMessage

# טעינת מפתחות
load_dotenv()
cohere_key = os.getenv("COHERE_API_KEY")

if not cohere_key:
    print("Error: COHERE_API_KEY is missing!")
else:
    llm = Cohere(model="command-r-08-2024", api_key=cohere_key) 
    print("Cohere LLM is ready for GEO Analysis!")

def ask_ai(prompt):
    """פונקציה לשליחת שאלה בודדת ל-AI"""
    try:
        messages = [ChatMessage(role="user", content=prompt)]
        response = llm.chat(messages)
        return response.message.content.strip()
    except Exception as e:
        print(f"API Error: {e}")
        return "ERROR"

def generate_strategic_summary(all_results):
    """סוכן AI שמפיק את מבנה ה-JSON המפורט עבור ה-Dashboard"""
    prompt = f"""
    נתח את ממצאי מחקר ה-GEO עבור 'ביטוח ישיר'. 
    עליך להחזיר רשימה (Array) של אובייקטים בפורמט JSON בלבד.
    
    עבור כל קטגוריה בנתונים: {json.dumps(all_results, ensure_ascii=False)}
    
    החזר JSON במבנה הבא לכל קטגוריה:
    {{
        "id": "ID הקטגוריה",
        "category_name": "שם הקטגוריה",
        "status": "קריטי / טעון שיפור / טוב",
        "stats": {{
            "positive_pct": "אחוז חיובי",
            "negative_pct": "אחוז שלילי",
            "competitor_pct": "אחוז אזכור מתחרים"
        }},
        "sources": {{
            "positive": ["דומיינים חיוביים"],
            "negative": ["דומיינים שליליים"]
        }},
        "ai_verdict": "משפט אחד על דעת ה-AI",
        "competitor_to_beat": {{ "name": "שם", "strength": "חוזקה" }},
        "content_gap": "מה חסר ברשת?",
        "action_item": "שורה תחתונה לשיווק",
        "full_report_url": "קישור לגוגל שיטס"
    }}
    """
    raw_res = ask_ai(prompt)
    try:
      start = raw_res.find('[')
      end = raw_res.rfind(']') + 1
      return json.loads(raw_res[start:end])
    except:
      return {"error": "Failed to generate structured plan"}

Cohere LLM is ready for GEO Analysis!


In [19]:
def ask_ai(prompt):
    try:
        messages = [ChatMessage(role="user", content=prompt)]
        response = llm.chat(messages)
        return response.message.content.strip()
    except Exception as e:
        print(f"API Error: {e}")
        return "ERROR"
    
    def generate_strategic_summary(all_results):
      """סוכן AI שמנתח את כלל הממצאים ומפיק תובנות רוחב לדשבורד"""
      prompt = f"""
      נתח את כל נתוני מחקר ה-GEO הבאים עבור המותג 'ביטוח ישיר'.
      הנתונים: {json.dumps(all_results, ensure_ascii=False)}
    
      החזר JSON בלבד (בלי הקדמות) בפורמט הבא:
      {{
          "overall_sentiment_score": "מספר בין 0-2",
          "market_position": "תקציר של 2 משפטים על המיצוב",
          "category_stats": [
              {{"id": "CAT_ID", "name": "שם", "score": 1.5, "main_competitor": "שם"}}
          ],
          "top_competitors": [
              {{"name": "שם חברה", "strength": "למה היא חזקה"}}
          ],
          "action_items": ["משימה 1", "משימה 2"]
      }}
      """
      raw_res = ask_ai(prompt)
      try:
          # חילוץ ה-JSON בבטחה
          start = raw_res.find('{')
          end = raw_res.rfind('}') + 1
          return json.loads(raw_res[start:end])
      except:
          return {"error": "Failed to parse strategic summary"}

In [20]:
# הגדרת הקטגוריות מה-JSON שקיבלת
CATEGORIES_JSON = {
    "CAT_01": {"name": "ביטוח רכב - אמינות", "focus": "אמינות המותג וחוזק החברה."},
    "CAT_02": {"name": "ביטוח טיסות - משתלם", "focus": "כדאיות בנסיעות לחו''ל וכיסוי רחב."},
    "CAT_03": {"name": "ביטוח דירה ומשכנתא", "focus": "תפיסת מומחיות בביטוחי מבנה ותכולה."},
    "CAT_04": {"name": "הביטוח הזול ביותר", "focus": "דומיננטיות במחיר נמוך."},
    "CAT_05": {"name": "יחס אנושי ונציגים", "focus": "איכות המענה, אמפתיה ואדיבות."},
    "CAT_06": {"name": "נוחות תפעולית ודיגיטל", "focus": "קלות רכישה ושימוש באתר/אפליקציה."},
    "CAT_07": {"name": "זמינות 24/7", "focus": "מהירות תגובה בחירום."},
    "CAT_08": {"name": "טיפול בתביעות", "focus": "מהימנות תשלום והגינות."},
    "CAT_09": {"name": "הביטוח המשתלם ביותר", "focus": "שילוב מחיר וכיסוי (Value for Money)."},
    "CAT_10": {"name": "חידוש ונאמנות", "focus": "כדאיות הישארות לאורך זמן."},
    "CAT_11": {"name": "מומחיות נציגים", "focus": "האם הנציג נתפס כיועץ מקצועי."},
    "CAT_12": {"name": "מצטרפים חדשים", "focus": "תמריצים ורושם ראשוני."}
}

with open("questions.json", "r", encoding="utf-8") as f:
    original_questions = json.load(f)

# איפוס המילון
categorized_dict = {cat_id: [] for cat_id in CATEGORIES_JSON.keys()}
categorized_dict["UNCATEGORIZED"] = []

print("מתחיל סיווג NLU למחקר GEO (לוגיקה משופרת)...")

for q in original_questions:
    # פרומפט פשוט וחד יותר ל-AI
    prompt_cat = f"""Classify this insurance question into ONE of these IDs: {', '.join(CATEGORIES_JSON.keys())}.
    
    Context:
    {json.dumps(CATEGORIES_JSON, ensure_ascii=False)}
    
    Question: {q}
    Answer only with the ID (e.g., CAT_01):"""
    
    raw_res = ask_ai(prompt_cat).upper()
    
    # חיפוש ה-ID בתוך התשובה (מטפל במקרים שה-AI עונה "The category is CAT_01")
    found_id = "UNCATEGORIZED"
    for cat_id in CATEGORIES_JSON.keys():
        if cat_id in raw_res:
            found_id = cat_id
            break
            
    categorized_dict[found_id].append({"שאלה": q, "מקור": "Original"})
    print(f"שאלה: {q[:30]}... -> תשובת AI: '{raw_res.strip()}' -> סווג סופית ל: {found_id}")

# ניקוי קטגוריות ריקות לסיכום
active_cats = {k: len(v) for k, v in categorized_dict.items() if len(v) > 0}
print(f"\nסיכום סיווגים: {active_cats}")

מתחיל סיווג NLU למחקר GEO (לוגיקה משופרת)...
שאלה: איזה ביטוח הכי זול לדירה?... -> תשובת AI: 'CAT_04' -> סווג סופית ל: CAT_04
שאלה: כמה עולה ביטוח הכי טוב לרכב?... -> תשובת AI: 'CAT_09' -> סווג סופית ל: CAT_09
שאלה: באיזו חברת ביטוח הכי כדאי להשת... -> תשובת AI: 'CAT_09' -> סווג סופית ל: CAT_09
שאלה: באיזה ביטוח שירות הלקוחות הכי ... -> תשובת AI: 'CAT_07' -> סווג סופית ל: CAT_07
שאלה: מה הביטוח הכי מקיף לבריאות איז... -> תשובת AI: 'CAT_09' -> סווג סופית ל: CAT_09

סיכום סיווגים: {'CAT_04': 1, 'CAT_07': 1, 'CAT_09': 3}


In [21]:
# --- תא 3: ג'ינרוט שאלות "פרסונה" ודילמות השוואתיות ---

print("מג'נרט שאלות ייעוץ מבוססות דילמות צרכניות...")

for cat_id, info in CATEGORIES_JSON.items():
    if cat_id == "UNCATEGORIZED": continue
    
    print(f"  מייצר סיטואציות עבור: {info['name']}...")
    
    # הפרומפט החדש: דורש מה-AI לייצר "קונפליקט" והשוואה בין חברות
    prompt_gen = f"""
    אתה מייצר שאלות עבור מחקר שוק. עליך לכתוב 5 דילמות של צרכנים ישראלים אמיתיים המתייעצים עם AI.
    הנושא: {info['name']}. הפוקוס: {info['focus']}.

    הנחיות קריטיות:
    1. השאלות חייבות להיות ספציפיות, "חופרות" ומחפשות המלצה על חברה ספציפית.
    2. כל שאלה חייבת לכלול התלבטות בין חברות (למשל: 'ביטוח ישיר מול ליברה', 'הפניקס לעומת ביטוח ישיר').
    3. השתמש בשפה טבעית של בני אדם (למשל: 'מי באמת משלם מהר?', 'האם השירות שלהם בטלפון שווה משהו?').
    4. אל תכתוב הקדמות. החזר רק את 5 השאלות בעברית, אחת בכל שורה.
    """
    
    response = ask_ai(prompt_gen)
    
    # עיבוד התשובות והכנסתן למאגר
    for line in response.split('\n'):
        clean_q = line.strip().lstrip('0123456789.- ')
        
        # סינון לוודא שהשאלה איכותית ומספיק ארוכה
        if len(clean_q) > 20 and "ERROR" not in clean_q:
            categorized_dict[cat_id].append({
                "שאלה": clean_q, 
                "מקור": "AI Expanded (Consumer Dilemma)"
            })

print("\n✅ המאגר הוגדל בהצלחה עם שאלות השוואתיות!")

מג'נרט שאלות ייעוץ מבוססות דילמות צרכניות...
  מייצר סיטואציות עבור: ביטוח רכב - אמינות...
  מייצר סיטואציות עבור: ביטוח טיסות - משתלם...
  מייצר סיטואציות עבור: ביטוח דירה ומשכנתא...
  מייצר סיטואציות עבור: הביטוח הזול ביותר...
  מייצר סיטואציות עבור: יחס אנושי ונציגים...
  מייצר סיטואציות עבור: נוחות תפעולית ודיגיטל...
  מייצר סיטואציות עבור: זמינות 24/7...
  מייצר סיטואציות עבור: טיפול בתביעות...
  מייצר סיטואציות עבור: הביטוח המשתלם ביותר...
  מייצר סיטואציות עבור: חידוש ונאמנות...
  מייצר סיטואציות עבור: מומחיות נציגים...
  מייצר סיטואציות עבור: מצטרפים חדשים...

✅ המאגר הוגדל בהצלחה עם שאלות השוואתיות!


In [22]:
# --- תא 4: ניתוח עומק אסטרטגי עם המלצות שיווקיות "תכלס" ---
import time
import json

print("🚀 מתחיל ניתוח GEO שיווקי: מחלץ המלצות לביצוע...")

for cat_id in categorized_dict:
    print(f"\n======= קטגוריה: {cat_id} ({CATEGORIES_JSON.get(cat_id, {}).get('name', 'כללי')}) =======")
    total_in_cat = len(categorized_dict[cat_id])
    
    for i, item in enumerate(categorized_dict[cat_id]):
        question = item["שאלה"]
        
        if "תשובת המודל" not in item or item["תשובת המודל"] == "ERROR":
            item["תשובת המודל"] = ask_ai(f"אתה יועץ ביטוח מומחה. ענה על שאלת הלקוח הבאה: {question}")
        
        answer = item["תשובת המודל"]
        print(f"  [{i+1}/{total_in_cat}] מפיק המלצה שיווקית...")
        
        # הפרומפט המעודכן שמתמקד ב"מה לעשות"
        analysis_prompt = f"""
        נתח את תשובת ה-AI הבאה כחוקר GEO שיווקי עבור 'ביטוח ישיר'.
        השאלה: {question}
        תשובת המודל: {answer}
        
        החזר JSON בלבד בעברית:
        {{
            "הזוכה_בסבב": "שם החברה המומלצת",
            "פעולה_שיווקית_מומלצת": "המלצה מעשית (למשל: לפרסם כתבה על X, להדגיש באתר Y)",
            "מסר_חסר": "איזה מסר ביטוח ישיר חייבת להוסיף לאתר כדי לנצח כאן?",
            "מדד_המלצה": 0-2
        }}
        """
        
        raw_analysis = ask_ai(analysis_prompt)
        
        try:
            start = raw_analysis.find('{')
            end = raw_analysis.rfind('}') + 1
            res = json.loads(raw_analysis[start:end])
            
            # שמירת הנתונים החדשים לתוך המילון
            item["הזוכה בסבב"] = res.get("הזוכה_בסבב", "לא הוכרע")
            item["פעולה שיווקית מומלצת"] = res.get("פעולה_שיווקית_מומלצת", "")
            item["מסר חסר לניצחון"] = res.get("מסר_חסר", "")
            item["ציון המלצה (0-2)"] = res.get("מדד_המלצה", 0)
            
        except Exception as e:
            item["שגיאת ניתוח"] = f"שגיאה: {str(e)}"
            print(f"    ⚠️ שגיאה בעיבוד שאלה {i+1}")
            
        time.sleep(10) # חשוב למניעת חסימה

print("\n✅ הניתוח הסתיים! כל שאלה מכילה כעת המלצה שיווקית.")

🚀 מתחיל ניתוח GEO שיווקי: מחלץ המלצות לביצוע...

======= קטגוריה: CAT_01 (ביטוח רכב - אמינות) =======
  [1/5] מפיק המלצה שיווקית...
  [2/5] מפיק המלצה שיווקית...
  [3/5] מפיק המלצה שיווקית...
  [4/5] מפיק המלצה שיווקית...
  [5/5] מפיק המלצה שיווקית...

======= קטגוריה: CAT_02 (ביטוח טיסות - משתלם) =======
  [1/5] מפיק המלצה שיווקית...
  [2/5] מפיק המלצה שיווקית...


KeyboardInterrupt: 

In [23]:
# --- תא 10: הפקת דו"ח ה-GEO השיווקי הסופי (גרסה מתוקנת ללא xlsxwriter) ---
import time
import json
import pandas as pd

# 1. פונקציית הסיכום המפורט (מייצרת את ה-JSON לדשבורד)
def generate_detailed_strategic_summary(all_results_by_cat):
    """
    סוכן AI שמנתח את תוצאות המחקר ומפיק Array של אובייקטים 
    לפי המבנה המדויק שנדרש.
    """
    # יצירת תקציר נתונים עבור הפרומפט כדי לא לחרוג ממגבלת ה-Tokens
    summary_for_ai = {}
    for cat_id, questions in all_results_by_cat.items():
        if not questions: continue
        # לוקחים מדגם של תשובות וציונים כדי שהסוכן יבין את המגמה
        sample = []
        for q in questions[:10]:
            sample.append({
                "שאלה": q.get("שאלה", ""),
                "הזוכה": q.get("הזוכה בסבב", "לא ידוע"),
                "סנטימנט": q.get("ציון המלצה (0-2)", 0)
            })
        summary_for_ai[cat_id] = sample

    prompt = f"""
    אתה מנהל אסטרטגיה שיווקית בכיר המומחה ב-AEO (אופטימיזציה למנועי AI).
    נתח את ממצאי מחקר ה-GEO עבור 'ביטוח ישיר' על בסיס הנתונים הבאים:
    {json.dumps(summary_for_ai, ensure_ascii=False)}
    
    עליך להחזיר רשימה (Array) של אובייקטים בפורמט JSON בלבד.
    לכל קטגוריה (CAT_XX), צור אובייקט במבנה הבא:
    {{
        "id": "ID הקטגוריה",
        "category_name": "שם הקטגוריה בעברית",
        "status": "קריטי / טעון שיפור / טוב",
        "stats": {{
            "positive_pct": "הערכת אחוז המלצות עלינו",
            "negative_pct": "הערכת אחוז המלצות על מתחרים",
            "competitor_pct": "אחוז שבו הוזכרנו לצד מתחרים"
        }},
        "sources": {{
            "positive": ["אתרים/מקורות שצוינו לטובה"],
            "negative": ["אתרים/מקורות שמהם מגיע תוכן שלילי"]
        }},
        "ai_verdict": "סיכום משפט אחד על תפיסת ה-AI",
        "competitor_to_beat": {{ "name": "המתחרה הדומיננטי", "strength": "מה החוזקה שלו?" }},
        "content_gap": "איזה מידע חסר ברשת כדי שה-AI ימליץ עלינו (למשל: נתונים מספריים, כתבות על שירות)?",
        "action_item": "משימה אחת ברורה לצוות השיווק לביצוע מיידי",
        "full_report_url": "PLACEHOLDER_LINK"
    }}
    """
    
    raw_res = ask_ai(prompt)
    try:
        start = raw_res.find('[')
        end = raw_res.rfind(']') + 1
        return json.loads(raw_res[start:end])
    except:
        print("⚠️ שגיאה בפענוח ה-JSON האסטרטגי. מחזיר תוצאה ריקה.")
        return []

print("📊 סוכן אסטרטגי מנתח מגמות ומפיק תובנות...")
marketing_report_array = generate_detailed_strategic_summary(categorized_dict)

# 2. שמירת ה-JSON המלא לדשבורד
dashboard_final_output = {
    "metadata": {
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_categories": len([k for k, v in categorized_dict.items() if v])
    },
    "categories_analysis": marketing_report_array,
    "raw_data": categorized_dict 
}

with open('dashboard_data.json', 'w', encoding='utf-8') as f:
    json.dump(dashboard_final_output, f, ensure_ascii=False, indent=4)
print("✅ קובץ dashboard_data.json מוכן במבנה האובייקטים המבוקש!")

# 3. שמירה לאקסל עם גיליונות נפרדים (תיקון השגיאה)
filename = "GEO_Marketing_Strategy_Direct_Insurance.xlsx"

try:
    # הסרת spec של ה-engine כדי להשתמש בברירת המחדל (openpyxl)
    with pd.ExcelWriter(filename) as writer:
        # גיליון סיכום אסטרטגי (השטחה של ה-JSON המורכב לטבלה)
        if marketing_report_array:
            summary_df = pd.json_normalize(marketing_report_array)
            summary_df.to_excel(writer, sheet_name="סיכום אסטרטגי", index=False)

        # גיליון נפרד לכל קטגוריה
        for cat_id, questions in categorized_dict.items():
            if not questions: continue
            
            df = pd.DataFrame(questions)
            # ניקוי שם הגיליון (אקסל מגביל ל-31 תווים)
            sheet_name = CATEGORIES_JSON.get(cat_id, {}).get("name", cat_id)[:30]
            sheet_name = "".join(c for c in sheet_name if c.isalnum() or c==' ')
            
            # בחירת עמודות רלוונטיות בלבד לאקסל
            cols = ["שאלה", "הזוכה בסבב", "פעולה שיווקית מומלצת", "מסר חסר לניצחון", "ציון המלצה (0-2)"]
            existing_cols = [c for c in cols if c in df.columns]
            
            df[existing_cols].to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✨ קובץ אקסל שיווקי נשמר בהצלחה ב: {filename}")
except Exception as e:
    print(f"❌ שגיאה נוספת בשמירת אקסל: {e}")
    print("טיפ: וודא שהקובץ לא פתוח באקסל בזמן הרצת הקוד.")
    
# # --- תא 6: הפקת דו"ח ה-GEO השיווקי הסופי ---
# # 1. איסוף כל הנתונים
# all_analyzed_data = []
# for questions in categorized_dict.values():
#     all_analyzed_data.extend(questions)

# # 2. הרצת הסוכן האסטרטגי שעדכנת בתא 1 (מייצר את ה-Battle Plan)
# print("📊 סוכן אסטרטגי בונה תוכנית תקיפה שיווקית...")
# marketing_report = generate_strategic_summary(all_analyzed_data)

# # 3. יצירת קובץ ה-JSON המלא לדשבורד (React)
# dashboard_final_output = {
#     "metadata": {
#         "date": time.strftime("%Y-%m-%d %H:%M:%S"),
#         "total_questions": len(all_analyzed_data)
#     },
#     "marketing_battle_plan": marketing_report, # כאן נכנסות ההמלצות "תכלס"
#     "raw_data": categorized_dict # כאן נכנס הפירוט לטבלאות
# }

# with open('dashboard_data.json', 'w', encoding='utf-8') as f:
#     json.dump(dashboard_final_output, f, ensure_ascii=False, indent=4)
# print("✅ קובץ dashboard_data.json מוכן עם המלצות שיווקיות!")

# # 4. שמירה לאקסל עם העמודות החדשות
# filename = "GEO_Marketing_Strategy_Direct_Insurance.xlsx"
# all_records = []
# for cat_id, data in categorized_dict.items():
#     df = pd.DataFrame(data)
#     if not df.empty:
#         df["קטגוריה"] = CATEGORIES_JSON.get(cat_id, {}).get("name", "כללי")
#         all_records.append(df)

# if all_records:
#     final_df = pd.concat(all_records, ignore_index=True)
#     # בחירת העמודות שיופיעו באקסל
#     cols_to_save = ["קטגוריה", "שאלה", "הזוכה בסבב", "פעולה שיווקית מומלצת", "מסר חסר לניצחון", "ציון המלצה (0-2)"]
#     existing_cols = [c for c in cols_to_save if c in final_df.columns]
#     final_df[existing_cols].to_excel(filename, index=False)

# print(f"✨ קובץ אקסל שיווקי נשמר ב: {filename}")

# # # --- תא 6: הפקת דו"ח ה-GEO השיווקי הסופי ---

# # # 1. איסוף כל הנתונים
# # all_analyzed_data = []
# # for questions in categorized_dict.values():
# #  all_analyzed_data.extend(questions)

# # # 2. הרצת הסוכן האסטרטגי שעדכנת בתא 1 (מייצר את ה-Battle Plan)
# # print("📊 סוכן אסטרטגי בונה תוכנית תקיפה שיווקית...")
# # marketing_report = generate_strategic_summary(all_analyzed_data)

# # # 3. יצירת קובץ ה-JSON המלא לדשבורד (React)
# # dashboard_final_output = {
# #     "metadata": {
# #         "date": time.strftime("%Y-%m-%d %H:%M:%S"),
# #         "total_questions": len(all_analyzed_data)
# #     },
# #     "marketing_battle_plan": marketing_report, # כאן נכנסות ההמלצות "תכלס"
# #     "raw_data": categorized_dict # כאן נכנס הפירוט לטבלאות
# # }

# # with open('dashboard_data.json', 'w', encoding='utf-8') as f:
# #     json.dump(dashboard_final_output, f, ensure_ascii=False, indent=4)
# # print("✅ קובץ dashboard_data.json מוכן עם המלצות שיווקיות!")

# # # 4. שמירה לאקסל עם העמודות החדשות
# # filename = "GEO_Marketing_Strategy_Direct_Insurance.xlsx"
# # all_records = []
# # for cat_id, data in categorized_dict.items():
# #     df = pd.DataFrame(data)
# #     if not df.empty:
# #         df["קטגוריה"] = CATEGORIES_JSON.get(cat_id, {}).get("name", "כללי")
# #         all_records.append(df)

# # if all_records:
# #     final_df = pd.concat(all_records, ignore_index=True)
# #     # בחירת העמודות שיופיעו באקסל
# #     cols_to_save = ["קטגוריה", "שאלה", "הזוכה בסבב", "פעולה שיווקית מומלצת", "מסר חסר לניצחון", "ציון המלצה (0-2)"]
# #     existing_cols = [c for c in cols_to_save if c in final_df.columns]
# #     final_df[existing_cols].to_excel(filename, index=False)

# # print(f"✨ קובץ אקסל שיווקי נשמר ב: {filename}")

📊 סוכן אסטרטגי מנתח מגמות ומפיק תובנות...
✅ קובץ dashboard_data.json מוכן במבנה האובייקטים המבוקש!
✨ קובץ אקסל שיווקי נשמר בהצלחה ב: GEO_Marketing_Strategy_Direct_Insurance.xlsx
